In [ ]:
import numpy as np
import cv2
import os
import json
import shutil
from sklearn.model_selection import train_test_split
import pydicom

In [ ]:
JSON_FILE_PATH = os.path.join("raw_data", "pneumonia-challenge-annotations-adjudicated-kaggle_2018.json")
SOURCE_IMAGES_DIR = os.path.join("raw_data", "pneumonia-challenge-dataset-adjudicated-kaggle_2018", "mdai_rsna_project_x9N20BZa_images_2018-07-20-153330")
OUTPUT_BASE_DIR = "data"

# Labels (Kaggle RSNA format)
LABEL_PNEUMONIA   = "L_v8n"            # Lung opacity
LABEL_NORMAL      = "L_o8w"            # Normal
LABEL_NO_OPACITY  = "L_yd0"            # No lung opacity / Not normal
NOT_PNEUMONIA_IDS = {LABEL_NORMAL, LABEL_NO_OPACITY}

# Dataset split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42

COPY_FILES = True

In [ ]:
def load_annotations(json_path: str) -> list:
    """
    Loads the JSON file and returns the list of annotations.
    
    Args:
        json_path (str): The path to the annotation JSON file.
        
    Returns:
        list: A list containing the parsed annotations.
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Annotation JSON file not found: {json_path}")
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    annotations = data["datasets"][0]["annotations"]
    print(f"Successfully loaded JSON. Total annotations: {len(annotations):,}")
    return annotations

In [ ]:
annotations = load_annotations(JSON_FILE_PATH)

In [ ]:
# study_records[StudyInstanceUID] = {
#     "sop"    : SOPInstanceUID    (used to locate the .dcm file name)
#     "series" : SeriesInstanceUID (used to navigate the nested DICOM folder layer)
#     "binary" : 1 (Pneumonia) | 0 (Not pneumonia) | None (not yet set)
# }

# }
 
study_records = {}

for ann in annotations:
    uid = ann.get("StudyInstanceUID")
    lid = ann.get("labelId")
    
    if not uid or not lid:
        continue

    # Initialize record if this is the first time we see this StudyInstanceUID
    if uid not in study_records:
        study_records[uid] = {"sop": None, "series": None, "binary": None}

    rec = study_records[uid]

    # Logic: 1 for Pneumonia, 0 for Normal/No-Opacity
    if lid == LABEL_PNEUMONIA:
        rec["binary"] = 1
        rec["sop"]    = ann.get("SOPInstanceUID")
        rec["series"] = ann.get("SeriesInstanceUID")
    elif lid in NOT_PNEUMONIA_IDS and rec["binary"] != 1:
        rec["binary"] = 0
        rec["sop"]    = ann.get("SOPInstanceUID")
        rec["series"] = ann.get("SeriesInstanceUID")

# Keep only studies that have a confirmed, calculated binary label
classified = {uid: rec for uid, rec in study_records.items() 
              if rec["binary"] is not None}

# Prepare lists for easier data splitting later
all_ids    = list(classified.keys())
all_labels = [classified[uid]["binary"] for uid in all_ids]
 
print("\n-----------------------------------------------------------")
print("Classification (calculated labels only):")
print("-----------------------------------------------------------")
print(f"  Pneumonia     (1) : {all_labels.count(1):>6,}")
print(f"  Not pneumonia (0) : {all_labels.count(0):>6,}")
print(f"  -----------------------------------")
print(f"  Total classified  : {len(all_ids):>6,}")
print(f"  Skipped           : {30000 - len(classified):>6,}  (no calculated label)")
print("-----------------------------------------------------------\n")


In [ ]:
def split_dataset(item_ids: list, item_labels: list, train_ratio: float = 0.70, val_ratio: float = 0.15, test_ratio: float = 0.15, seed: int = 42) -> dict:
    """
    Splits a dataset into train, validation, and test sets
    
    Args:
        item_ids (list): A list of unique identifiers for the data samples (e.g., filenames, UIDs).
        item_labels (list): A list of corresponding class labels for stratification.
        train_ratio (float): Proportion of the dataset to include in the train split.
        val_ratio (float): Proportion of the dataset to include in the validation split.
        test_ratio (float): Proportion of the dataset to include in the test split.
        seed (int): Random seed for reproducibility.
        
    Returns:
        dict: A dictionary containing the splits ('train', 'val', 'test') with their respective IDs and labels.
              Format: {'train': (ids_list, labels_list), ...}
    """
    # Ensure ratios sum to 1.0 (with a small tolerance for floating point errors)
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-5, "Split ratios must sum to 1.0"

    # First, separate the Train set, the remaining goes to a temporary set
    train_ids, temp_ids, train_labels, temp_labels = train_test_split(
        item_ids, item_labels, 
        test_size=(val_ratio + test_ratio), 
        stratify=item_labels, 
        random_state=seed
    )

    # Split the temporary set into Validation and Test sets based on their relative ratio
    val_test_ratio = test_ratio / (val_ratio + test_ratio) 
    val_ids, test_ids, val_labels, test_labels = train_test_split(
        temp_ids, temp_labels, 
        test_size=val_test_ratio, 
        stratify=temp_labels, 
        random_state=seed
    )

    print("Splits created successfully:")
    print(f"  Train set size: {len(train_ids):>6,} items")
    print(f"  Val set size:   {len(val_ids):>6,} items")
    print(f"  Test set size:  {len(test_ids):>6,} items\n")
    
    return {
        "train": (train_ids, train_labels),
        "val":   (val_ids, val_labels),
        "test":  (test_ids, test_labels)
    }

In [ ]:
dataset_splits = split_dataset(all_ids, all_labels, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO, seed=RANDOM_SEED)

In [ ]:
def index_files(root_dir: str, extension: str = ".dcm") -> dict:
    """
    General purpose file indexer. Maps filenames (without extension) to absolute paths.
    
    Args:
        root_dir (str): The directory to search.
        extension (str): The file extension to look for (default: .dcm).
        
    Returns:
        dict: Mapping of {filename_without_extension: absolute_path}.
    """
    if not os.path.exists(root_dir):
        raise FileNotFoundError(f"Directory not found: {root_dir}")

    file_map = {}
    
    for root, _, filenames in os.walk(root_dir):
        for f in filenames:
            if f.endswith(extension):
                # Get ID (filename without extension)
                file_id = os.path.splitext(f)[0]
                
                # Get absolute path with Windows long-path prefix
                full_path = os.path.abspath(os.path.join(root, f))
                if os.name == 'nt' and not full_path.startswith("\\\\?\\"):
                    full_path = "\\\\?\\" + full_path
                    
                file_map[file_id] = full_path

    return file_map

In [ ]:
def organize_dataset(splits_dict: dict, id_to_file_map: dict, id_to_label_map: dict, 
                     label_to_folder: dict, output_base: str):
    """
    General purpose dataset organizer. Copies files based on splits and labels.
    
    Args:
        splits_dict: {'train': ([ids], [labels]), ...}
        id_to_file_map: {id: source_path}
        id_to_label_map: {id: label_value}
        label_to_folder: {label_value: "folder_name"}
        output_base: base output directory
    """
    copied_count = 0
    
    for split_name, (split_ids, _) in splits_dict.items():
        for uid in split_ids:
            # Determine the label for this specific ID
            label = id_to_label_map[uid]
            folder_name = label_to_folder[label]
            
            # Construct destination
            target_dir = os.path.abspath(os.path.join(output_base, split_name, folder_name))
            if os.name == 'nt' and not target_dir.startswith("\\\\?\\"):
                target_dir = "\\\\?\\" + target_dir
            
            os.makedirs(target_dir, exist_ok=True)
            
            # Copy the file from source to target
            source_file = id_to_file_map.get(uid)

            if source_file and os.path.exists(source_file):
                # Get the original extension
                _, extension = os.path.splitext(source_file)
                
                target_file = os.path.join(target_dir, f"{uid}{extension}")
                
                shutil.copy2(source_file, target_file)
                copied_count += 1

                if copied_count % 2000 == 0:
                    print(f"  - Copied {copied_count:,} files...")
                
    print(f"Finished! Organized {copied_count} files into {output_base}.")

In [ ]:
if COPY_FILES:
    raw_file_map = index_files(SOURCE_IMAGES_DIR, extension=".dcm")

    study_to_file_map = {}
    for study_uid, rec in classified.items():
        sop_id = rec["sop"]
        if sop_id in raw_file_map:
            study_to_file_map[study_uid] = raw_file_map[sop_id]
        
    label_map = {uid: rec["binary"] for uid, rec in classified.items()}
    folder_map = {0: "not_pneumonia", 1: "pneumonia"}

    organize_dataset(
        splits_dict=dataset_splits, 
        id_to_file_map=study_to_file_map, 
        id_to_label_map=label_map,
        label_to_folder=folder_map,
        output_base=OUTPUT_BASE_DIR
    )
else:
    print("File copying is disabled. Skipping the index and copy operation.")